# 03 · Embed — 02 OpenAI embeddings (text-embedding-3-large, 3072 dim)

This is the real model a production corpus would be built with. The key is read straight from the environment.

If `OPENAI_API_KEY` isn't set, this notebook reports that plainly and skips rather than crashing. The offline path in `01-offline-embeddings.ipynb` has already proven the wiring works with no key at all — this notebook is what you run once you have one.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `get_embedding_dimension` | Returns the fixed dimension for the configured embedding model. | `get_embedding_dimension()` → `3072` |
| `embed_texts` | Embeds a list of texts with `text-embedding-3-large`, batching requests. | `embed_texts(sample_chunks)` |
| `embed_query` | Embeds a single query string. | `embed_query("what converts nutrients into ATP?")` |
| `write_index_manifest` | Writes the JSON record of what model, dimension, and tokenizer built an index. | `write_index_manifest(path, index_name=..., model=..., dimension=3072, ...)` |


## Step 1 — bootstrap the repo path and confirm the environment

Jupyter starts this kernel with the notebook's own directory as `cwd`, so `nbio` has to be located and put on `sys.path` before anything else can import it. `env` also captures whether `OPENAI_API_KEY` is set, for the guard check later.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio

repo_root = nbio.bootstrap()
env = nbio.show_environment()

## Step 2 — pin the embedding model and its dimension

`text-embedding-3-large` is hardcoded here rather than made configurable — a config knob that lets the model drift from the dimension recorded in an index manifest is the bug this whole stage exists to prevent (see `03-dimensions.ipynb`).

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIMENSION = 3072  # cloud corpora built with this model are pinned to this dimension

## Step 3 — define `get_embedding_dimension`

A single accessor for the dimension pinned above, so callers never hardcode `3072` themselves.

In [ ]:
def get_embedding_dimension() -> int:
    """Returns the fixed dimension for the configured embedding model."""
    return EMBEDDING_DIMENSION

## Step 4 — define the OpenAI client helper

Reads `OPENAI_API_KEY` straight from the environment and raises if it isn't set — this is the guard the next steps rely on.

In [ ]:
def _get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not set")
    from openai import OpenAI
    return OpenAI(api_key=api_key)

## Step 5 — define `embed_texts`, the batched embedding call

In [ ]:
import os

_LAST_USAGE: dict = {}


def embed_texts(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    """Embed texts with text-embedding-3-large, reading the API key
    directly from the environment.
    """
    client = _get_openai_client()
    all_embeddings: list[list[float]] = []
    total_prompt_tokens = 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
        total_prompt_tokens += getattr(response.usage, "prompt_tokens", 0)
    _LAST_USAGE["model"] = EMBEDDING_MODEL
    _LAST_USAGE["prompt_tokens"] = total_prompt_tokens
    _LAST_USAGE["completion_tokens"] = 0  # embeddings have no completion side
    return all_embeddings


## Step 6 — define `embed_query`

A single-string convenience wrapper around `embed_texts`, for the one-query-at-a-time case a search does at runtime.

In [ ]:
def embed_query(query: str) -> list[float]:
    return embed_texts([query])[0]

## Step 7 — check the guard: is `OPENAI_API_KEY` set?

Before ever calling the real API, confirm plainly what this notebook will do next — skip cleanly, or actually embed. This is the guard demonstrated on its own, before the real embedding call in the next step.

In [ ]:
has_key = bool(env.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY is set" if has_key else "OPENAI_API_KEY not set, skipping the real embedding call below")

## Step 8 — embed the sample chunks under a spend ceiling (or skip cleanly if no key)

Same five sample chunks as `01-offline-embeddings.ipynb`, so the two notebooks are directly comparable: same input, two different embedding paths. This is the one real paid call in this notebook, so it runs inside `nbio.cost_meter` — 50 cents is enough for many times this batch size on `text-embedding-3-large`, sized to catch a mistake rather than ration a normal run.

In [ ]:
sample_chunks = [
    "The mitochondria is the powerhouse of the cell, converting nutrients into ATP through oxidative phosphorylation.",
    "Retrieval-augmented generation grounds a language model's answer in retrieved passages rather than parametric memory alone.",
    "A vector embedding maps a passage of text onto a point in a high-dimensional space so that semantic similarity becomes geometric distance.",
]

with nbio.cost_meter(budget_usd=0.50) as meter:
    if not has_key:
        print(
            "OPENAI_API_KEY not set, skipping -- the offline path in "
            "01-offline-embeddings.ipynb has already proven the wiring works. "
            "Set OPENAI_API_KEY and re-run this cell for real 3072-dim vectors."
        )
    else:
        vectors = embed_texts(sample_chunks)
        meter.record(_LAST_USAGE["model"], _LAST_USAGE["prompt_tokens"], _LAST_USAGE["completion_tokens"])
        print(f"{len(vectors)} vectors, dimension {len(vectors[0])}")
        assert len(vectors[0]) == get_embedding_dimension()

print()
print(meter.report())


## Step 9 — define `write_index_manifest`

Without a manifest, a set of saved indexes can accumulate with nothing recording what model or dimension built any of them. `write_index_manifest()` is the one JSON file that turns "why is retrieval returning nothing" back into a one-line dimension check instead of a debugging session. See `03-dimensions.ipynb` for what happens without it.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path


def write_index_manifest(
    path: Path,
    *,
    index_name: str,
    model: str,
    dimension: int,
    tokenizer: str,
    max_tokens: int,
    overlap: int,
) -> dict:
    """Write the one JSON file that keeps 'index built at dimension X,
    queried with a model at dimension Y' from looking like an empty corpus
    instead of a config mismatch.
    """
    manifest = {
        "index_name": index_name,
        "model": model,
        "dimension": dimension,
        "tokenizer": tokenizer,
        "max_tokens": max_tokens,
        "overlap": overlap,
        "built_at": datetime.now(timezone.utc).isoformat(),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2))
    return manifest

## Step 10 — write this index's manifest and read it back

In [ ]:
manifest_path = repo_root / "runs" / "demo-embed-openai" / "index_manifest.json"
manifest = write_index_manifest(
    manifest_path,
    index_name="demo-openai-3072",
    model=EMBEDDING_MODEL,
    dimension=EMBEDDING_DIMENSION,
    tokenizer="cl100k_base",  # tiktoken encoding used by text-embedding-3-large
    max_tokens=8191,
    overlap=0,
)
print(f"wrote {manifest_path}")
nbio.show_json(manifest)

`runs/` is gitignored at the repo root, so this manifest never travels with a commit — it's a local record of what you actually built, read back before you ever send a query at an index.